# LoRA Fine-tuning 공통 스켈레톤

Language·Vision·Speech 중 **하나**를 선택합니다. TODO를 완성하기 전에는 전체 실행이 되지 않습니다. 태스크별 모델·데이터·전처리·학습 설정은 직접 구현합니다.

Colab GPU 런타임에서 작업합니다. 동일한 Test 데이터로 학습 전후를 비교하고, 최소 5개 예측을 기록합니다.

## 1. 과제 설정

**태스크: 벽바둑 Ⅱ "한 수로 이기는 수" 찾기 (Language, 생성형)**

넷플릭스 「데블스 플랜: 데드 어게인」 4일차 게임 **벽바둑 Ⅱ**의 국면을 텍스트로 보여 주고, 둘 차례인 플레이어가 **한 수(말 이동 + 벽 설치)로 즉시 이기는 수**를 답하게 합니다. 체스의 '1수 메이트' 퍼즐과 같은 구조입니다.

| 구분 | 내용 |
|---|---|
| 입력 | 둘 차례 + 말 좌표 목록 + 7×7 판(말·벽)을 그린 텍스트 + 엔진이 계산한 합법 수 목록 |
| 출력 | `출발-도착 벽방향` 형식의 수 한 줄 (예: `c3-c5 S`) |
| 정답 판정 | 채점용 게임 엔진이 수를 실제로 적용해 **즉시 종료되고 내 영역이 더 넓으면 정답**. 승리 수가 여러 개면 그중 아무거나 정답 |
| 지표 | Solve rate(승리 수를 낸 비율). 제공된 Accuracy 코드에 정답=1, 예측=승리 여부(1/0)를 넣어 계산 |

In [ ]:
TASK = "Language"  # Language / Vision / Speech
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "nahanah/wallgo2-win-in-1"
SEED = 42
OUTPUT_DIR = "./assignment_runs/wallgo"

**선택 이유**

[모델](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) · [데이터](https://huggingface.co/datasets/nahanah/wallgo2-win-in-1)

- **게임 — 벽바둑 Ⅱ**: 시즌 3 게임 중 유일하게 **모든 정보가 공개되고 운이 없는** 턴제 추상전략 게임입니다. 다른 게임(양자택일, 미스터리 거래소, 보석 암시장, 경매 빙고 등)은 비공개 정보·심리전이 핵심이라 정답이 하나로 정해지지 않지만, 벽바둑은 규칙만으로 "이기는 수"를 검증할 수 있어 Accuracy형 평가가 가능합니다. 같은 계열 게임(시즌 2 벽바둑)은 AlphaZero 기반 연구(WallZero, 2026)로도 분석된 바 있어 분석 가치가 있습니다.
- **데이터 — 직접 만든 합성 퍼즐(Hub 공개)**: 벽바둑 데이터셋이 Hub에 없어서, 규칙 엔진과 랜덤 자가대국 3,600판으로 퍼즐 5,640개를 만들어 Hub에 공개했습니다. 정답은 사람이 아니라 엔진이 모든 합법 수를 직접 적용해 계산하므로 라벨 오류가 없습니다. 같은 대국에서 나온 국면은 같은 분할에 넣었습니다(그룹 분할).
- **모델 — Qwen2.5-0.5B-Instruct**: 수를 '생성'해야 하므로 decoder-only 생성 모델을 골랐습니다. 약 5억 Parameters로 Colab T4에서 LoRA 학습이 가능하고, 대화형(Instruct) 모델이라 학습 전에도 지시문으로 답을 시도할 수 있어 **Prompting baseline**을 측정할 수 있습니다. 라이선스는 Apache 2.0입니다.
- **실행 과정 기록**: 처음 시도 때는 무료 Colab의 GPU 사용 한도에 걸려 GPU를 할당받지 못했습니다. CPU용으로 더 작은 모델(SmolLM2-135M)과 데이터 일부로 줄여 시도했지만, 학습 전 평가에만 약 15분이 걸리고 학습 중 CPU 런타임 RAM이 부족해 세션이 다운되었습니다. 다음 날 GPU가 다시 할당되어 원래 설계(Qwen2.5-0.5B, Train 전체)로 T4에서 최종 실행했습니다. T4에서의 1차 실행(판 격자만 입력, 2 Epoch)은 학습 전후 Solve rate가 모두 0%였고, 학습 후에도 553/569개가 빈 칸에서 출발하는 등 규칙 위반 수였습니다. 모델이 글자 격자에서 말의 위치를 읽지 못한다고 판단해, 2차 실행에서는 입력에 엔진이 계산한 말 좌표 목록과 합법 수 목록을 추가해 다시 학습했습니다. 아래 출력은 모두 이 2차 실행의 결과입니다.
- **새 Classification head 없음**: 생성 모델의 기존 출력층(언어 모델 head)을 그대로 쓰므로, 학습 전후 변화를 새 head가 아닌 LoRA 학습의 효과로 해석할 수 있습니다.

## 2. 실행 환경

아래 설치 코드는 공통 라이브러리를 설치하는 기본 예시입니다. 먼저 설치 셀과 import 셀을 실행하여 정상 작동하는지 확인합니다. 버전 충돌이 발생하면 오류 메시지와 공식 문서를 참고하여 설치할 버전을 조정합니다.


In [ ]:
# Colab에 이미 설치된 패키지는 그대로 두고 없는 것만 설치합니다(-U를 쓰지 않아 GPU용 torch 조합이 바뀌지 않음).
%pip install -q transformers datasets peft accelerate pandas
# Colab 런타임에 따라 기본 torchao(0.10.0)가 있으면 PEFT의 최소 요구 버전보다 낮아 get_peft_model에서 ImportError가 납니다(없으면 아래처럼 "not installed" 경고만 나옵니다).
# 이 실습은 양자화를 쓰지 않으므로 torchao를 제거합니다.
%pip uninstall -y -q torchao

In [ ]:
import importlib.metadata
import torch
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, set_seed
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from IPython.display import display

import os, gc
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # 메모리 조각화 완화
torch.set_num_threads(os.cpu_count())  # CPU 코어를 모두 사용
set_seed(SEED)
print("torch 스레드 수:", torch.get_num_threads())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
for package in ("torch", "transformers", "datasets", "peft", "accelerate"):
    print(package, importlib.metadata.version(package))

torch 스레드 수: 2
GPU: Tesla T4
torch 2.14.1+cu130
transformers 5.22.1
datasets 4.8.5
peft 0.21.2
accelerate 1.15.0


## 3. 데이터 로딩과 분할

load_dataset으로 데이터를 불러와 Dataset 또는 DatasetDict를 반환합니다. 분할 이름이 다르면 train / validation / test로 맞춥니다. 제공된 함수가 기존 분할을 유지하고 없는 분할만 생성합니다. 분할이 없으면 80% / 10% / 10%로 나눕니다.

분할 결과의 크기와 클래스 분포를 확인합니다. 같은 화자·원본을 묶어야 하는 데이터에는 무작위 분할 대신 그룹별 분할을 적용합니다.

**이 데이터의 분할**: Hub에 이미 train / validation / test가 있으므로 제공된 함수는 기존 분할을 그대로 유지합니다. 같은 대국에서 나온 국면은 서로 비슷하므로, 업로드 단계에서 **대국(game_id) 단위로 묶어** 나눴습니다(스켈레톤이 말하는 '그룹별 분할'). 아래에서 대국·국면이 분할 간에 겹치지 않는지 확인합니다.

아래 셀에는 **채점용 게임 엔진**도 들어 있습니다. 모델이 낸 수를 실제로 적용해 승패를 판정하는 데 씁니다.

In [ ]:
# ===== 채점용 벽바둑 Ⅱ 게임 엔진 =====
"""벽바둑 Ⅱ (데블스 플랜: 데드 어게인, 2인전 규칙) 게임 엔진.

규칙
- 7x7 판. 행 1~7(위→아래), 열 a~g(왼→오).
- 플레이어 R(빨강), B(파랑). 각자 사각 말 2개, 원형 말 2개.
  표기: 사각 말 = 대문자(R, B), 원형 말 = 소문자(r, b).
- 사각 말: 상하좌우 한 방향으로 1~2칸 직선 이동.
- 원형 말: 상하좌우 1칸 이동 후, 원하면 왼쪽/오른쪽으로 꺾어 1칸 더 이동.
- 제자리 이동 불가. 벽이나 다른 말이 있으면 그 방향으로 지나갈 수 없음.
- 이동한 말의 상하좌우 경계 중 비어 있는 곳(판 가장자리 제외)에 벽 1개를 반드시 설치.
- 모든 영역(벽으로 나뉜 칸 덩어리)이 한 플레이어의 말만 포함하면 즉시 종료.
- 점수 = 내 말만 있는 영역의 칸 수 합. 동률이면 가장 큰 단일 영역 비교, 그래도 같으면 무승부.
- 2라운드(2인전) 규칙이므로 벽 부수기 없음.
"""
from collections import deque
import json

N = 7
COLS = "abcdefg"
DIRS = {"N": (-1, 0), "S": (1, 0), "W": (0, -1), "E": (0, 1)}
LEFT_RIGHT = {"N": ("W", "E"), "S": ("E", "W"), "W": ("S", "N"), "E": ("N", "S")}
PLAYERS = ("R", "B")


def sq(r, c):
    return f"{COLS[c]}{r + 1}"


def parse_sq(s):
    return int(s[1:]) - 1, COLS.index(s[0])


def edge(r1, c1, r2, c2):
    """인접한 두 칸 사이 경계의 고유 키."""
    a, b = sorted([(r1, c1), (r2, c2)])
    return (a, b)


class State:
    def __init__(self, pieces, walls, to_move):
        # pieces: dict (r, c) -> (player, kind)  kind: 'S' 사각 / 'C' 원형
        self.pieces = dict(pieces)
        self.walls = set(walls)
        self.to_move = to_move

    def copy(self):
        return State(self.pieces, self.walls, self.to_move)

    # ---------- 이동 ----------
    def can_step(self, r, c, d):
        dr, dc = DIRS[d]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < N and 0 <= nc < N):
            return None
        if edge(r, c, nr, nc) in self.walls or (nr, nc) in self.pieces:
            return None
        return nr, nc

    def destinations(self, r, c):
        kind = self.pieces[(r, c)][1]
        dests = set()
        for d in DIRS:
            first = self.can_step(r, c, d)
            if first is None:
                continue
            dests.add(first)
            if kind == "S":
                second = self.can_step(*first, d)
                if second:
                    dests.add(second)
            else:
                for t in LEFT_RIGHT[d]:
                    second = self.can_step(*first, t)
                    if second:
                        dests.add(second)
        return dests

    def wall_options(self, r, c):
        opts = []
        for d, (dr, dc) in DIRS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < N and 0 <= nc < N and edge(r, c, nr, nc) not in self.walls:
                opts.append(d)
        return opts

    def legal_moves(self):
        """(출발, 도착, 벽방향) 목록. 출발 칸을 비운 상태에서 도착 칸 벽 옵션을 계산."""
        moves = []
        for (r, c), (p, _) in sorted(self.pieces.items()):
            if p != self.to_move:
                continue
            for dest in sorted(self.destinations(r, c)):
                for w in self.wall_options(*dest):
                    moves.append(((r, c), dest, w))
        return moves

    def apply(self, move):
        (r, c), (nr, nc), w = move
        s = self.copy()
        s.pieces[(nr, nc)] = s.pieces.pop((r, c))
        dr, dc = DIRS[w]
        s.walls.add(edge(nr, nc, nr + dr, nc + dc))
        s.to_move = PLAYERS[1 - PLAYERS.index(self.to_move)]
        return s

    # ---------- 영역 ----------
    def regions(self):
        seen, out = set(), []
        for r in range(N):
            for c in range(N):
                if (r, c) in seen:
                    continue
                q, cells = deque([(r, c)]), []
                seen.add((r, c))
                while q:
                    cr, cc = q.popleft()
                    cells.append((cr, cc))
                    for dr, dc in DIRS.values():
                        nr, nc = cr + dr, cc + dc
                        if 0 <= nr < N and 0 <= nc < N and (nr, nc) not in seen \
                                and edge(cr, cc, nr, nc) not in self.walls:
                            seen.add((nr, nc))
                            q.append((nr, nc))
                owners = {self.pieces[x][0] for x in cells if x in self.pieces}
                out.append((cells, owners))
        return out

    def result(self):
        """종료가 아니면 None. 종료면 dict(winner, score)."""
        regs = self.regions()
        if any(len(o) > 1 for _, o in regs):
            return None
        score = {p: sum(len(cells) for cells, o in regs if o == {p}) for p in PLAYERS}
        biggest = {p: max([len(cells) for cells, o in regs if o == {p}] or [0]) for p in PLAYERS}
        a, b = PLAYERS
        if score[a] != score[b]:
            winner = a if score[a] > score[b] else b
        elif biggest[a] != biggest[b]:
            winner = a if biggest[a] > biggest[b] else b
        else:
            winner = None
        return {"winner": winner, "score": score}

    def winning_moves(self):
        legal = self.legal_moves()
        wins = [m for m in legal if (res := self.apply(m).result()) and res["winner"] == self.to_move]
        return wins, legal

    # ---------- 텍스트 ----------
    def to_text(self):
        lines = ["  " + " ".join(COLS)]
        for r in range(N):
            row = f"{r + 1} "
            for c in range(N):
                p = self.pieces.get((r, c))
                ch = "." if p is None else (p[0] if p[1] == "S" else p[0].lower())
                sep = "|" if c < N - 1 and edge(r, c, r, c + 1) in self.walls else " "
                row += ch + (sep if c < N - 1 else "")
            lines.append(row.rstrip())
            if r < N - 1:
                under = "  " + " ".join("-" if edge(r, c, r + 1, c) in self.walls else " " for c in range(N))
                lines.append(under)
        return "\n".join(lines)

    def to_json(self):
        return json.dumps({
            "pieces": [[sq(r, c), p, k] for (r, c), (p, k) in sorted(self.pieces.items())],
            "walls": sorted([f"{sq(*a)}-{sq(*b)}" for a, b in self.walls]),
            "to_move": self.to_move,
        })

    @staticmethod
    def from_json(s):
        d = json.loads(s)
        pieces = {parse_sq(x): (p, k) for x, p, k in d["pieces"]}
        walls = set()
        for w in d["walls"]:
            a, b = w.split("-")
            walls.add(edge(*parse_sq(a), *parse_sq(b)))
        return State(pieces, walls, d["to_move"])


def move_to_str(m):
    (r, c), (nr, nc), w = m
    return f"{sq(r, c)}-{sq(nr, nc)} {w}"


def str_to_move(s):
    """'c3-c5 S' 형식. 형식이 틀리면 None."""
    import re
    m = re.search(r"\b([a-g][1-7])\s*-\s*([a-g][1-7])\s+([NSEW])\b", s)
    if not m:
        return None
    return parse_sq(m.group(1)), parse_sq(m.group(2)), m.group(3)


def judge(state_json, answer_text):
    """모델 답을 채점: 'win' / 'legal_loss' / 'illegal' / 'format_error'."""
    s = State.from_json(state_json)
    mv = str_to_move(answer_text)
    if mv is None:
        return "format_error"
    if mv not in s.legal_moves():
        return "illegal"
    res = s.apply(mv).result()
    return "win" if res and res["winner"] == s.to_move else "legal_loss"


# ===== 데이터 로딩과 분할 =====
from datasets import Dataset, DatasetDict

def prepare_splits(data, seed=SEED):
    splits = DatasetDict({"train": data}) if isinstance(data, Dataset) else DatasetDict(data)
    if "train" not in splits:
        raise ValueError("학습용 분할의 이름을 train으로 지정하세요.")
    if "test" not in splits:
        split = splits["train"].train_test_split(test_size=0.1, seed=seed)
        splits["train"], splits["test"] = split["train"], split["test"]
        validation_ratio = 1 / 9
    else:
        validation_ratio = 0.1
    if "validation" not in splits:
        split = splits["train"].train_test_split(test_size=validation_ratio, seed=seed)
        splits["train"], splits["validation"] = split["train"], split["test"]
    return DatasetDict({name: splits[name] for name in ("train", "validation", "test")})

def load_data():
    """반환: Dataset 또는 train을 포함한 DatasetDict."""
    return load_dataset(DATASET_ID)

raw = prepare_splits(load_data())
print(raw)
print(raw["train"][0])

# 분할 크기·대국 수·난이도·누출 점검
for name, data in raw.items():
    df = data.to_pandas()
    print(f"{name}: 퍼즐 {len(df)}개, 대국 {df.game_id.nunique()}개, 평균 합법 수 {df.n_legal.mean():.1f}, "
          f"평균 승리 수 {df.n_win.mean():.2f}, 랜덤 합법 수 기대 정답률 {(df.n_win / df.n_legal).mean():.2%}")
games = {k: set(v["game_id"]) for k, v in raw.items()}
states = {k: set(v["state"]) for k, v in raw.items()}
print("대국 겹침(train∩val, train∩test, val∩test):",
      len(games["train"] & games["validation"]), len(games["train"] & games["test"]), len(games["validation"] & games["test"]))
print("동일 국면 겹침:", len(states["train"] & states["validation"]), len(states["train"] & states["test"]), len(states["validation"] & states["test"]))

# 라벨 품질 점검: 저장된 정답 수를 엔진으로 다시 채점
check = raw["train"].select(range(300))
print("정답 수 재채점 결과:", pd.Series([judge(s, a) for s, a in zip(check["state"], check["answer"])]).value_counts().to_dict())
print("\n예시 퍼즐 (차례:", raw["test"][0]["to_move"], ")")
print(raw["test"][0]["board"])
print("승리 수:", raw["test"][0]["winning_moves"])

DatasetDict({
    train: Dataset({
        features: ['id', 'game_id', 'turn', 'to_move', 'board', 'state', 'answer', 'winning_moves', 'n_legal', 'n_win'],
        num_rows: 4538
    })
    validation: Dataset({
        features: ['id', 'game_id', 'turn', 'to_move', 'board', 'state', 'answer', 'winning_moves', 'n_legal', 'n_win'],
        num_rows: 533
    })
    test: Dataset({
        features: ['id', 'game_id', 'turn', 'to_move', 'board', 'state', 'answer', 'winning_moves', 'n_legal', 'n_win'],
        num_rows: 569
    })
})
{'id': 'p000000', 'game_id': 'g00000', 'turn': 27, 'to_move': 'B', 'board': '  a b c d e f g\n1 . . . . . . .\n          -    \n2 . . . . b . .\n  -            \n3 . . . . . .|.\n  - - -       -\n4 R r|b|. B|.|.\n  -     -      \n5 .|R|. . .|.|.\n  - - -     -  \n6 . . .|.|. r .\n            - -\n7 . . .|.|B . .', 'state': '{"pieces": [["e2", "B", "C"], ["a4", "R", "S"], ["b4", "R", "C"], ["c4", "B", "C"], ["e4", "B", "S"], ["b5", "R", "S"], ["f6", "R", "C"], [

## 4. 모델과 전처리

Tokenizer / Image processor / Audio feature extractor 등을 선택합니다. 새 Classification head를 쓰면 초기화 사실과 레이블 매핑을 기록합니다. map으로 모델 입력과 labels를 구성합니다.

- Language: 텍스트·레이블 처리. 생성형 방식은 Prompt와 정답 구간의 Loss 처리를 결정합니다.
- Vision: 이미지 크기·정규화 처리
- Speech: Sampling rate·길이 처리

추론에서도 같은 전처리 의미를 유지합니다. Test 데이터에 학습용 무작위 증강을 적용하지 않습니다.

In [ ]:
SYSTEM_PROMPT = (
    "You solve puzzles of the board game WallGo II. Board 7x7, columns a-g (left to right), rows 1-7 (top to bottom). "
    "Players R and B. Uppercase = square piece (moves 1-2 cells straight in one direction), lowercase = circle piece "
    "(moves 1 cell, then may turn left or right for 1 more cell). Pieces and walls block movement. "
    "After moving, place one wall on a side of the moved piece (N=up, S=down, W=left, E=right). "
    "In the board, '|' is a wall between left/right cells and '-' in the gap lines is a wall between up/down cells. "
    "When every region contains pieces of only one player, the game ends and the larger total territory wins."
)

def legal_move_list(example):
    """엔진으로 계산한 합법 수 목록(어느 수가 이기는지에 대한 정보는 없음)."""
    return [move_to_str(m) for m in State.from_json(example["state"]).legal_moves()]

def piece_list(example):
    """말 위치를 좌표 글자로 풀어 쓴 목록. 격자에서 공백을 세지 않아도 위치를 바로 알 수 있게 합니다."""
    s = State.from_json(example["state"])
    me = example["to_move"]
    kind = {"S": "square", "C": "circle"}
    mine = [f"{sq(r, c)} {kind[k]}" for (r, c), (p, k) in sorted(s.pieces.items(), key=lambda x: (x[0][1], x[0][0])) if p == me]
    opp = [f"{sq(r, c)} {kind[k]}" for (r, c), (p, k) in sorted(s.pieces.items(), key=lambda x: (x[0][1], x[0][0])) if p != me]
    return mine, opp

def build_messages(example):
    # 1차 실행(판 격자만 제공)에서는 모델이 격자에서 말 위치를 읽지 못해 거의 모든 답이 규칙 위반이었습니다.
    # 2차 실행: (1) 말 좌표 목록으로 위치를 명시하고 (2) 합법 수 목록을 주어 답을 그 안에서 고르게 하며
    # (3) 모델은 후보 중 '즉시 이기는 수'를 찾는 데 집중합니다.
    mine, opp = piece_list(example)
    user = (f"To move: {example['to_move']}\n"
            f"Your pieces ({example['to_move']}): {', '.join(mine)}\n"
            f"Opponent pieces: {', '.join(opp)}\n"
            f"Board:\n{example['board']}\n\n"
            f"Legal moves: {', '.join(legal_move_list(example))}\n\n"
            f"Which legal move wins immediately for {example['to_move']}? "
            "Answer with exactly one move from the list, in the format 'from-to wall'.")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

def build_prompt(example, processor):
    return processor.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True)

def load_model_and_processor():
    """반환: (이미 학습된 모델, 저장 가능한 Tokenizer 또는 Processor)."""
    processor = AutoTokenizer.from_pretrained(MODEL_ID)
    processor.padding_side = "right"  # 학습용. 생성(추론) 때만 left로 바꿉니다.
    if processor.pad_token is None:  # 배치 패딩용 토큰이 없으면 종료 토큰을 사용
        processor.pad_token = processor.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
    return model, processor

model, processor = load_model_and_processor()
END_ID = processor.convert_tokens_to_ids("<|im_end|>")

def preprocess(example):
    """입력: 원본 샘플 dict. 반환: 모델 입력과 labels를 포함한 dict."""
    prompt_ids = processor(build_prompt(example, processor), add_special_tokens=False)["input_ids"]
    answer_ids = processor(example["answer"], add_special_tokens=False)["input_ids"] + [END_ID]
    input_ids = prompt_ids + answer_ids
    # 프롬프트 구간은 -100으로 가려 Loss에서 제외하고, 정답 수(+종료 토큰) 구간만 학습합니다.
    labels = [-100] * len(prompt_ids) + answer_ids
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

prepared = raw.map(preprocess, remove_columns=raw["train"].column_names)
print({k: prepared["train"][0][k][-12:] for k in ("input_ids", "labels")})

lengths = pd.Series([len(x) for x in prepared["train"]["input_ids"]])
answer_lengths = pd.Series([sum(l != -100 for l in x) for x in prepared["train"]["labels"]])
print("입력 토큰 길이:", lengths.describe().round(1).to_dict())
print("Loss 대상(정답) 토큰 수:", answer_lengths.describe().round(1).to_dict())
print("\n프롬프트 예시:\n" + build_prompt(raw["test"][0], processor))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'input_ids': [4427, 151645, 198, 151644, 77091, 198, 68, 19, 5655, 21, 468, 151645], 'labels': [-100, -100, -100, -100, -100, -100, 68, 19, 5655, 21, 468, 151645]}
입력 토큰 길이: {'count': 4538.0, 'mean': 465.9, 'std': 49.1, 'min': 382.0, '25%': 429.0, '50%': 457.0, '75%': 494.0, 'max': 759.0}
Loss 대상(정답) 토큰 수: {'count': 4538.0, 'mean': 6.0, 'std': 0.0, 'min': 6.0, '25%': 6.0, '50%': 6.0, '75%': 6.0, 'max': 6.0}

프롬프트 예시:
<|im_start|>system
You solve puzzles of the board game WallGo II. Board 7x7, columns a-g (left to right), rows 1-7 (top to bottom). Players R and B. Uppercase = square piece (moves 1-2 cells straight in one direction), lowercase = circle piece (moves 1 cell, then may turn left or right for 1 more cell). Pieces and walls block movement. After moving, place one wall on a side of the moved piece (N=up, S=down, W=left, E=right). In the board, '|' is a wall between left/right cells and '-' in the gap lines is a wall between up/down cells. When every region contains pieces of o

**전처리·레이블 매핑·Classification head 초기화 여부**

- **Instruction / Input / Output 설계**: system 메시지에 게임 규칙과 판 표기법(Instruction)을, user 메시지에 차례·말 좌표 목록·판 텍스트(Input)를 넣고, assistant 응답은 `c3-c5 S` 같은 수 한 줄(Output)로 정했습니다. user 메시지에는 **말 좌표 목록**(예: `Your pieces (B): c4 circle, d3 circle, ...`)과 엔진이 계산한 **합법 수 목록**도 넣었습니다. 1차 실행에서 모델이 판 격자에서 말의 위치를 읽지 못해 규칙 위반 수만 냈기 때문입니다. 합법 수 목록에는 어느 수가 이기는지에 대한 정보가 없으므로, 과제는 "후보 중 즉시 이기는 수 고르기"가 되고 랜덤 기준(합법 수 중 무작위 선택)과 그대로 비교할 수 있습니다. 모델의 대화 템플릿(`apply_chat_template`)을 그대로 사용해 학습과 추론의 입력 형식을 맞췄습니다. 같은 규칙 설명도 한국어보다 영어가 토큰 수가 적어 입력 길이와 학습 시간을 줄일 수 있으므로, 프롬프트는 영어로 작성했습니다.
- **판 표기**: 대문자=사각 말, 소문자=원형 말, `|`=좌우 벽, 행 사이 줄의 `-`=위아래 벽, `.`=빈 칸. 판을 글자 격자로 써야 Tokenizer가 토큰으로 바꿀 수 있습니다(텍스트 → 토큰 → 임베딩).
- **정답 구간 Loss**: 프롬프트 토큰의 labels는 -100으로 가려 Loss에서 빼고, **정답 수와 종료 토큰(`<|im_end|>`)에만 Loss**를 줍니다. SFT의 Loss = −log p(정답 토큰)이며, 문제 지문을 외우는 대신 답을 내는 동작만 학습합니다.
- **정답이 여러 개인 경우**: 학습 정답(`answer`)은 승리 수 중 점수 차가 가장 큰 수 하나입니다. 평가는 엔진으로 하므로 다른 승리 수를 내도 정답으로 처리합니다.
- **Classification head**: 새로 만들지 않았습니다. 생성 모델의 기존 언어 모델 head를 그대로 사용합니다.

## 5. 학습 전 평가

pipeline 추론 결과를 정답과 같은 형식의 레이블 목록으로 변환합니다. 예: 정답이 0/1이라면 예측도 0/1로 변환합니다. 생성형 모델이 유효하지 않은 답을 출력하면 정답과 일치하지 않는 레이블로 처리하고 평가에서 제외하지 않습니다.

아래 두 TODO를 완성하면 Accuracy와 결과 표는 자동 계산됩니다. Test 전체에서 평가하며, 정답을 추론 입력에 포함하지 않습니다. 학습 후에도 같은 데이터 순서·전처리·추론 설정을 사용합니다. 분류 외 태스크는 지표를 변경합니다.


In [ ]:
INFERENCE_KWARGS = {"max_new_tokens": 10, "do_sample": False, "repetition_penalty": 1.0,
                    "return_full_text": False, "batch_size": 16}
LAST_EVAL = {}

def get_targets(test_data):
    """반환: Test 순서대로 정답 레이블을 담은 list."""
    # 모든 퍼즐에는 승리 수가 있으므로 '이기는 수를 낸다(1)'가 정답 레이블입니다.
    return [1] * len(test_data)

def predict_labels(current_model, processor, test_data):
    """반환: Test 순서대로 예측 레이블을 담은 list."""
    current_model.eval()
    base = current_model.get_base_model() if isinstance(current_model, PeftModel) else current_model
    processor.padding_side = "left"  # 생성 모델은 왼쪽 패딩으로 배치 추론
    generator = pipeline("text-generation", model=base, tokenizer=processor,
                         device=0 if torch.cuda.is_available() else -1)
    prompts = [build_prompt(ex, processor) for ex in test_data]
    outputs = generator(prompts, **INFERENCE_KWARGS)
    processor.padding_side = "right"
    texts = [o[0]["generated_text"].strip() for o in outputs]
    # 엔진으로 채점: win / legal_loss(합법이지만 못 이김) / illegal(규칙 위반) / format_error(형식 오류)
    verdicts = [judge(s, t) for s, t in zip(test_data["state"], texts)]
    LAST_EVAL.clear()
    LAST_EVAL.update(outputs=texts, verdicts=verdicts)
    del generator, outputs
    gc.collect()  # RAM 정리
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # 평가 후 GPU 메모리 반환(학습 단계 OOM 방지)
    # 승리 수면 1, 그 밖의 모든 출력(형식 오류 포함)은 0으로 처리하고 평가에서 제외하지 않습니다.
    return [1 if v == "win" else 0 for v in verdicts]

def accuracy(targets, predictions):
    if len(targets) == 0 or len(targets) != len(predictions):
        raise ValueError("정답과 예측은 길이가 같고 비어 있지 않아야 합니다.")
    return sum(a == b for a, b in zip(targets, predictions)) / len(targets)

def evaluate_model(current_model, processor, test_data):
    targets = get_targets(test_data)
    predictions = predict_labels(current_model, processor, test_data)
    if len(targets) != len(test_data):
        raise ValueError("Test의 모든 샘플에 대해 정답과 예측을 반환하세요.")
    metrics = {"accuracy": accuracy(targets, predictions)}
    table = pd.DataFrame({"sample_id": range(len(targets)), "target": targets,
                          "prediction": predictions})
    return metrics, table

before_metrics, before = evaluate_model(model, processor, raw["test"])
before_detail = dict(LAST_EVAL)
random_baseline = sum(n / m for n, m in zip(raw["test"]["n_win"], raw["test"]["n_legal"])) / len(raw["test"])
print("학습 전 Solve rate:", before_metrics)
print(f"비교 기준 — 랜덤 합법 수의 기대 정답률: {random_baseline:.2%}")
print("추론 설정:", INFERENCE_KWARGS)
print("학습 전 출력 유형:", pd.Series(before_detail["verdicts"]).value_counts().to_dict())
display(before.head())
display(pd.DataFrame({"sample_id": range(5), "puzzle_id": raw["test"]["id"][:5],
                      "정답 예시": raw["test"]["answer"][:5], "학습 전 출력": before_detail["outputs"][:5],
                      "판정": before_detail["verdicts"][:5]}))

학습 전 Solve rate: {'accuracy': 0.054481546572934976}
비교 기준 — 랜덤 합법 수의 기대 정답률: 11.81%
추론 설정: {'max_new_tokens': 10, 'do_sample': False, 'repetition_penalty': 1.0, 'return_full_text': False, 'batch_size': 16}
학습 전 출력 유형: {'format_error': 296, 'legal_loss': 180, 'illegal': 62, 'win': 31}


,sample_id,target,prediction
0,0,1,0
1,1,1,0
2,2,1,0
3,3,1,0
4,4,1,0


,sample_id,puzzle_id,정답 예시,학습 전 출력,판정
0,0,p000003,d3-e4 S,d3-c2 wall,format_error
1,1,p000004,d4-e5 S,f3-d3 W,illegal
2,2,p000005,d4-e5 S,d3-d1 wall,format_error
3,3,p000013,c3-b3 N,c3-c2,format_error
4,4,p000014,c3-b3 N,c3-c2,format_error


## 6. LoRA 적용

모델의 모듈 이름을 확인한 뒤 target_modules, r, lora_alpha 등을 설정합니다. task_type은 해당 PEFT 지원 방식에 맞게 선택합니다. Classification head를 학습해야 한다면 modules_to_save 등 적절한 방법으로 포함하고, 저장 대상도 확인합니다.

In [ ]:
# 모듈 이름 확인: Qwen2의 Attention은 q_proj·k_proj·v_proj·o_proj, MLP는 gate_proj·up_proj·down_proj입니다.
module_names = sorted({name.rsplit(".", 1)[-1] for name, _ in model.named_modules()})
print("모듈 이름:", module_names)

def make_lora_config(model):
    """반환: 선택한 모델에 맞는 LoraConfig."""
    targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    names = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
    assert set(targets) <= names, "모듈 이름을 model.named_modules()로 확인하세요."
    return LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                      target_modules=targets, bias="none")

lora_config = make_lora_config(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = [name for name, param in model.named_parameters() if param.requires_grad]
print("학습 대상 Parameter 텐서 수:", len(trainable))
print("LoRA 대상 예시:", trainable[:4])
print("LoRA가 아닌 학습 대상:", [n for n in trainable if "lora_" not in n])

모듈 이름: ['', '0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '3', '4', '5', '6', '7', '8', '9', 'act_fn', 'down_proj', 'embed_tokens', 'gate_proj', 'input_layernorm', 'k_proj', 'layers', 'lm_head', 'mlp', 'model', 'norm', 'o_proj', 'post_attention_layernorm', 'q_proj', 'rotary_emb', 'self_attn', 'up_proj', 'v_proj']
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
학습 대상 Parameter 텐서 수: 336
LoRA 대상 예시: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight']
LoRA가 아닌 학습 대상: []


**LoRA 설정 이유 및 실제 학습 대상 모듈**

- `task_type=CAUSAL_LM`: 다음 토큰을 예측하는 생성 모델용 PEFT 방식입니다.
- `target_modules`: 24개 층 각각의 **Attention(q·k·v·o_proj)과 MLP(gate·up·down_proj)** 전체에 LoRA를 넣었습니다. 감성 분석처럼 이미 아는 능력의 '출력 형식'만 맞추는 일이 아니라, 판을 읽고 영역을 따지는 **새로운 판단 규칙**을 익혀야 하므로, 토큰 사이 정보를 모으는 Attention뿐 아니라 지식을 저장·처리하는 MLP에도 변화량을 허용했습니다.
- `r=16`, `lora_alpha=32`(scale 2), `lora_dropout=0.05`: Query·Value만 쓰는 기본값(r=8)보다 표현 가능한 변화의 폭을 넓혔습니다. r은 변화량 BA가 가질 수 있는 독립 방향 수의 상한일 뿐 성능을 보장하지는 않으며, Test 성능으로 고르지 않았습니다.
- 기존 가중치 W는 모두 고정하고 A·B만 학습합니다. 새 head가 없으므로 LoRA가 아닌 학습 대상은 없습니다(위 출력으로 확인). 실제 학습 가능한 Parameters 수와 비율은 `print_trainable_parameters()` 출력에 있습니다.

## 7. 학습

TrainingArguments, Data collator, Trainer를 구성합니다. Train은 학습에, Validation은 설정 선택에 사용합니다. Test는 전달하지 않습니다. Padding·labels 및 Loss가 태스크와 맞는지 확인합니다.

학습률 2e-4, 2 Epoch, Batch size 2 × Gradient accumulation 8(유효 16)과 gradient checkpointing을 썼습니다. 합법 수 목록으로 입력이 길어져 Batch 8·4에서는 T4(15GB) 메모리가 부족했기 때문입니다. 스케줄은 cosine과 warmup 3%, fp16 혼합 정밀도로 설정합니다. `DataCollatorForSeq2Seq`가 input_ids는 pad 토큰으로, labels는 -100으로 패딩해 패딩 위치가 Loss에 들어가지 않게 합니다. 매 Epoch마다 Validation Loss를 확인하고, 체크포인트는 저장하지 않으며 최종 Epoch의 모델을 평가합니다.

In [ ]:
def make_trainer(model, processor, train_data, validation_data):
    """반환: train() 메서드를 가진 Trainer. OUTPUT_DIR을 저장 경로로 사용합니다."""
    args = TrainingArguments(
        output_dir=OUTPUT_DIR, learning_rate=2e-4, num_train_epochs=2,
        per_device_train_batch_size=2, per_device_eval_batch_size=2, gradient_accumulation_steps=8,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},  # T4 메모리 절약
        lr_scheduler_type="cosine", warmup_steps=0.03, weight_decay=0.0,
        eval_strategy="epoch", save_strategy="no", logging_strategy="steps", logging_steps=25,
        fp16=torch.cuda.is_available(), label_names=["labels"], remove_unused_columns=False,
        report_to="none", seed=SEED)
    collator = DataCollatorForSeq2Seq(processor, padding=True, label_pad_token_id=-100)
    return Trainer(model=model, args=args, train_dataset=train_data,
                   eval_dataset=validation_data, data_collator=collator)

print("학습에 사용한 Train:", len(prepared["train"]), "/ Validation:", len(prepared["validation"]))
trainer = make_trainer(model, processor, prepared["train"], prepared["validation"])
trainer.train()
print(pd.DataFrame([x for x in trainer.state.log_history if "eval_loss" in x])[["epoch", "eval_loss"]])

학습에 사용한 Train: 4538 / Validation: 533


Epoch,Training Loss,Validation Loss
1,0.403168,0.423941
2,0.351349,0.406806


   epoch  eval_loss
0    1.0   0.423941
1    2.0   0.406806


## 8. 학습 후 평가

학습 전 평가와 동일한 함수·Test 데이터를 사용합니다. 성능 향상이 없거나 실패한 사례도 그대로 기록합니다.

In [ ]:
after_metrics, after = evaluate_model(model, processor, raw["test"])
assert len(before) >= 5 and len(after) >= 5, "비교 샘플을 최소 5개 기록하세요."
assert before["sample_id"].is_unique and after["sample_id"].is_unique
assert before[["sample_id", "target"]].reset_index(drop=True).equals(
    after[["sample_id", "target"]].reset_index(drop=True)
), "학습 전후 동일한 샘플·순서·정답을 사용하세요."
comparison = before.rename(columns={"prediction": "before"}).merge(
    after[["sample_id", "prediction"]].rename(columns={"prediction": "after"}),
    on="sample_id", validate="one_to_one"
)
display(pd.DataFrame({"before": before_metrics, "after": after_metrics}))
display(comparison)

after_detail = dict(LAST_EVAL)
delta_pp = (after_metrics["accuracy"] - before_metrics["accuracy"]) * 100
print(f"동일한 Test {len(comparison)}개 Solve rate: {before_metrics['accuracy']:.2%} → {after_metrics['accuracy']:.2%} "
      f"({delta_pp:+.2f}%p) / 랜덤 합법 수 기대값 {random_baseline:.2%}")
print("실패→성공:", int(((comparison.before == 0) & (comparison.after == 1)).sum()),
      "/ 성공→실패:", int(((comparison.before == 1) & (comparison.after == 0)).sum()))

verdict_table = pd.DataFrame({"학습 전": pd.Series(before_detail["verdicts"]).value_counts(),
                              "학습 후": pd.Series(after_detail["verdicts"]).value_counts()}).fillna(0).astype(int)
print("출력 유형별 개수 (win=승리 수, legal_loss=합법이지만 못 이김, illegal=규칙 위반, format_error=형식 오류)")
display(verdict_table)

detail = pd.DataFrame({"sample_id": range(len(raw["test"])), "puzzle_id": raw["test"]["id"],
                       "n_legal": raw["test"]["n_legal"], "n_win": raw["test"]["n_win"],
                       "정답 예시": raw["test"]["answer"],
                       "학습 전 출력": before_detail["outputs"], "학습 전 판정": before_detail["verdicts"],
                       "학습 후 출력": after_detail["outputs"], "학습 후 판정": after_detail["verdicts"]})
detail["난이도(합법 수)"] = pd.cut(detail.n_legal, [0, 10, 15, 20, 100], labels=["≤10", "11-15", "16-20", "21+"])
print("난이도별 학습 후 Solve rate")
display(detail.assign(성공=detail["학습 후 판정"] == "win").groupby("난이도(합법 수)", observed=True)["성공"].agg(["count", "mean"]))
print("비교 예시 (처음 8개)")
display(detail.head(8))
print("학습 후 실패 사례 (최대 5개)와 판")
for _, r in detail[detail["학습 후 판정"] != "win"].head(5).iterrows():
    ex = raw["test"][int(r.sample_id)]
    print(f"--- {r.puzzle_id} / 차례 {ex['to_move']} / 모델: {r['학습 후 출력']} ({r['학습 후 판정']}) / 승리 수: {ex['winning_moves']}")
    print(ex["board"])

,before,after
accuracy,0.054482,0.195079


,sample_id,target,before,after
0,0,1,0,0
1,1,1,0,0
2,2,1,0,0
3,3,1,0,0
4,4,1,0,0
...,...,...,...,...
564,564,1,1,0
565,565,1,1,0
566,566,1,1,0
567,567,1,0,0


동일한 Test 569개 Solve rate: 5.45% → 19.51% (+14.06%p) / 랜덤 합법 수 기대값 11.81%
실패→성공: 99 / 성공→실패: 19
출력 유형별 개수 (win=승리 수, legal_loss=합법이지만 못 이김, illegal=규칙 위반, format_error=형식 오류)


,학습 전,학습 후
format_error,296,0
illegal,62,1
legal_loss,180,457
win,31,111


난이도별 학습 후 Solve rate


,count,mean
난이도(합법 수),,
≤10,159,0.333333
11-15,151,0.172185
16-20,119,0.159664
21+,140,0.092857


비교 예시 (처음 8개)


,sample_id,puzzle_id,n_legal,n_win,정답 예시,학습 전 출력,학습 전 판정,학습 후 출력,학습 후 판정,난이도(합법 수)
0,0,p000003,27,3,d3-e4 S,d3-c2 wall,format_error,d3-c2 N,legal_loss,21+
1,1,p000004,12,2,d4-e5 S,f3-d3 W,illegal,f3-g3 N,legal_loss,11-15
2,2,p000005,13,1,d4-e5 S,d3-d1 wall,format_error,d3-d2 W,legal_loss,11-15
3,3,p000013,16,1,c3-b3 N,c3-c2,format_error,c3-c2 N,legal_loss,16-20
4,4,p000014,16,1,c3-b3 N,c3-c2,format_error,c3-c2 N,legal_loss,16-20
5,5,p000015,15,1,c3-b3 N,c3-c2 S,legal_loss,c3-c2 N,legal_loss,11-15
6,6,p000016,14,2,c3-b3 N,c3-c2,format_error,c3-c2 N,legal_loss,11-15
7,7,p000024,13,3,f2-f4 S,d5 square,format_error,d5-c4 S,legal_loss,11-15


학습 후 실패 사례 (최대 5개)와 판
--- p000003 / 차례 B / 모델: d3-c2 N (legal_loss) / 승리 수: ['c4-d4 E', 'd3-d4 E', 'd3-e4 S']
  a b c d e f g
1 . . . . . .|.
               
2 . . . . . .|.
          - -  
3 . .|.|b . B .
  - -     - - -
4 R r|b . .|B|r
  -   - -   -  
5 . . . .|.|R|.
              -
6 . . .|. .|.|.
        -     -
7 . . . .|.|. .
--- p000004 / 차례 B / 모델: f3-g3 N (legal_loss) / 승리 수: ['d4-e4 S', 'd4-e5 S']
  a b c d e f g
1 . . . . . .|.
               
2 . . . . . .|.
          - -  
3 . .|.|. . B .
  - -   - - - -
4 R .|b b .|B|r
  -   - -   -  
5 r . . .|.|R|.
  -           -
6 . . .|. .|.|.
        -     -
7 . . . .|.|. .
--- p000005 / 차례 B / 모델: d3-d2 W (legal_loss) / 승리 수: ['d4-e5 S']
  a b c d e f g
1 . . . . . .|.
               
2 . . . . . .|.
          - -  
3 . .|.|B|. . .
  - -   - - - -
4 R .|b b .|B|.
  -   - -   - -
5 r . . .|.|R|r
  -           -
6 . . .|. .|.|.
        -     -
7 . . . .|.|. .
--- p000013 / 차례 R / 모델: c3-c2 N (legal_loss) / 승리 수: ['c3-b3 N']
  a b c d

## 9. 저장과 분석

Adapter를 저장합니다. 재사용에는 호환되는 Base model과 Processor·설정도 필요합니다. 추가 학습한 Classification head가 있다면 함께 저장되는지 확인합니다.

In [ ]:
from pathlib import Path

adapter_dir = Path(OUTPUT_DIR) / "adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
model.config.save_pretrained(adapter_dir)
print("Base model:", MODEL_ID)
print("Adapter·Processor·모델 설정 저장 위치:", adapter_dir.resolve())
print("저장된 파일:", sorted(p.name for p in adapter_dir.iterdir()))

Base model: Qwen/Qwen2.5-0.5B-Instruct
Adapter·Processor·모델 설정 저장 위치: /content/assignment_runs/wallgo/adapter
저장된 파일: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'config.json', 'tokenizer.json', 'tokenizer_config.json']


### 결과 분석

- **Base model 이름·Revision과 저장 위치**: `Qwen/Qwen2.5-0.5B-Instruct`(Hub 기본 `main` Revision, Apache 2.0). LoRA Adapter와 Tokenizer·모델 설정은 `./assignment_runs/wallgo/adapter`(Colab 기준 `/content/assignment_runs/wallgo/adapter`)에 저장했습니다. 재사용하려면 같은 Base model을 불러온 뒤 `PeftModel.from_pretrained`로 Adapter를 붙이고, 입력은 같은 프롬프트 형식(말 좌표 목록 + 판 + 합법 수 목록)으로 만듭니다.
- **데이터 규모·분할, 평가 지표와 학습 설정**
  - 데이터: 직접 만든 `nahanah/wallgo2-win-in-1`(랜덤 자가대국 3,600판에서 뽑은 퍼즐 5,640개). Train 4,538(대국 2,493) / Validation 533 / Test 569(대국 312). 대국 단위로 나눠 분할 간 대국·국면 겹침은 0이며, 정답 재채점 300/300이 승리였습니다.
  - 지표: Solve rate = 모델이 낸 수를 엔진에 적용했을 때 즉시 이기는 비율(스켈레톤 Accuracy에 정답=1, 예측=승리 여부 1/0). 형식 오류도 0으로 세어 제외하지 않았습니다. 정답이 항상 1이므로 Accuracy = 예측(승리 여부)의 평균 = Solve rate입니다.
  - 비교 기준(Test): 합법 수 중 무작위 선택 **11.81%**(위 출력). 참고로 같은 엔진·같은 Test로 노트북 밖에서 따로 계산한 "목록의 첫 번째 수만 고르기"는 12.13%, "마지막 수만 고르기"는 12.48%입니다.
  - 학습: LoRA r=16, alpha=32, dropout 0.05, 24개 층의 q·k·v·o·gate·up·down_proj. 학습 Parameters 8,798,208개 / 전체 502,830,976개(**1.75%**). 학습률 2e-4, 2 Epoch, Batch 2 × Gradient accumulation 8(유효 16), gradient checkpointing, cosine, fp16, T4 GPU. Loss는 정답 수와 종료 토큰에만 걸었습니다.
- **학습 전후 변화와 실패 사례**
  - 동일한 Test 569개에서 Solve rate **5.45% → 19.51%(+14.06%p)**. 실패→성공 99개, 성공→실패 19개입니다. 학습 후 성능은 세 가지 단순 기준(11.8~12.5%)보다 약 7%p 높아, 목록 위치를 외운 것이 아니라 국면에 따라 이기는 수를 어느 정도 골라낸다고 볼 수 있습니다.
  - 출력 유형: 형식 오류 296 → **0**, 규칙 위반 62 → **1**, 합법이지만 못 이김 180 → 457, 승리 31 → **111**. 학습 전 모델은 `d3-c2 wall`, `d5 square`처럼 지시문의 단어를 그대로 쓰거나 형식을 지키지 못했는데, 학습 후에는 거의 모든 답이 목록 안의 합법 수가 됐습니다. 남은 실패(457개)는 대부분 '규칙상 가능하지만 이기지 못하는 수'입니다.
  - Validation Loss 0.424(1 Epoch) → 0.407(2 Epoch). 1차 실행(판 격자만 입력, 아래 참고)의 1.19~1.23보다 크게 낮습니다.
  - 난이도(합법 수 개수)별 학습 후 Solve rate: ≤10개 33.3% · 11–15개 17.2% · 16–20개 16.0% · 21개 이상 9.3%. 후보가 많을수록 성능이 떨어지지만, 같은 구간의 랜덤 기대값(19.4% · 11.8% · 8.3% · 6.2%, 같은 엔진·같은 Test로 노트북 밖에서 계산)보다는 모든 구간에서 높습니다.
  - 실패 사례: p000013·p000014·p000015·p000016은 비슷한 국면이고 정답이 모두 `c3-b3 N`인데, 모델은 네 문제 모두 `c3-c2 N`을 냈습니다. 같은 말을 고르고 벽 방향(N)도 맞혔지만 도착 칸 한 칸이 틀려 영역이 닫히지 않았습니다. p000003에서도 승리 수 `d3-e4 S`의 말(d3)은 골랐지만 `d3-c2 N`으로 다른 칸에 갔습니다. 즉 '어느 말을 움직일지'는 비교적 잘 고르지만, '어디로 가서 어느 벽을 세워야 영역이 완전히 갈리는지'라는 마지막 계산에서 자주 틀립니다.
- **결과 해석에 영향을 준 조건**
  - 새 Classification head는 없습니다. 기존 언어 모델 head를 그대로 썼으므로 변화는 LoRA 학습의 효과입니다.
  - **입력 표현 변경(1차 → 2차 실행)**: 1차 실행에서는 판을 글자 격자로만 주었고, 결과는 학습 전후 모두 0%였습니다(학습 후 569개 중 553개가 빈 칸에서 출발하는 규칙 위반). 모델이 공백과 `|` 사이에서 글자 위치를 세어 말의 좌표를 알아내지 못한 것입니다. 2차 실행에서는 (1) 말 좌표 목록과 (2) 엔진이 계산한 합법 수 목록을 넣었습니다. 합법 수 목록에는 어느 수가 이기는지에 대한 정보가 없으므로 과제는 '후보 중 즉시 이기는 수 고르기'가 되고, 랜덤 선택 기준과 공정하게 비교할 수 있습니다. 대신 '판을 보고 규칙상 가능한 수를 스스로 만드는 능력'은 평가하지 않습니다.
  - **Test 사용에 대한 한계**: 입력 형식을 바꾼 결정은 1차 실행의 Test 출력(0%, 규칙 위반 553개)을 보고 내렸습니다. 규칙상 Test는 설정 선택에 쓰지 않아야 하므로 이 점을 한계로 밝힙니다. 바꾼 것은 입력 형식 한 가지뿐이고, r·학습률·Epoch 같은 학습 설정은 Test 결과로 고르지 않았습니다(Batch와 Epoch는 메모리와 시간 때문에 정했습니다). 1차 실행의 Validation Loss(1.19~1.23)도 높게 남아 같은 문제를 보여 주었습니다. 더 엄밀하게 하려면 이런 진단을 Validation 출력으로만 해야 합니다.
  - 학습 전 평가의 5.45%는 형식 오류(296개)의 영향이 큽니다. 학습 전 모델의 '판단력'만 보면 더 높을 수 있으므로, 학습 효과의 상당 부분은 '목록에서 정확히 하나를 골라 형식대로 쓰는 법'을 배운 것입니다. 그래서 학습 후 성능은 학습 전 수치보다 랜덤·첫 수 기준과 비교하는 것이 더 공정합니다.
  - 학습 정답은 여러 승리 수 중 점수 차가 가장 큰 하나뿐이라, 다른 승리 수를 내도 Loss에서는 틀린 것으로 처리됩니다(평가는 엔진이라 정답 처리).
  - 무료 Colab GPU 한도와 T4 메모리 때문에 Batch를 줄였고, 설정 탐색 없이 2 Epoch 한 번만 학습했습니다.
- **한계와 개선 방향**
  - 19.5%는 랜덤보다 낫지만 여전히 5문제 중 4문제는 틀립니다. '한 수로 이기기'는 수를 둔 뒤 영역이 어떻게 나뉘는지 계산해야 하는 여러 단계의 추론인데, 모델은 이를 중간 풀이 없이 한 번에 답해야 합니다. 후보마다 '두면 영역이 닫히는가'를 먼저 쓰게 하는 중간 풀이 학습이나, 후보별 승패를 분류하는 방식으로 바꾸면 개선 여지가 있습니다.
  - 데이터: 판을 회전·대칭(8가지)한 퍼즐로 라벨 오류 없이 데이터를 8배로 늘리고, 승리 수가 여러 개인 퍼즐은 모든 승리 수를 정답 샘플로 넣을 수 있습니다.
  - 모델: 1.5B 이상 모델, 더 많은 Epoch, Validation으로 학습률 선택.
  - 평가: 학습 전 모델은 `max_new_tokens=10`과 엄격한 형식 검사 때문에 형식 오류로 처리된 답이 많았을 수 있습니다(학습 전후 같은 설정). 또 Test 안에 같은 대국에서 나온 비슷한 국면(예: p000013~p000016)이 있어 표본이 완전히 독립적이지는 않습니다.
  - 입력 형식: 이번에 만든 '말 좌표 목록 + 판 + 합법 수 목록'을 표준 입력으로 정해 두면, 판 사진 같은 다른 형식도 이 텍스트로만 바꿔 주면 같은 모델을 쓸 수 있습니다. 판을 글자 격자만으로 읽게 하는 것은 0.5B 모델에게 어려웠고, 이미지를 직접 읽게 하려면 비전-언어 모델이 필요합니다.
  - 정리: LoRA는 전체의 1.75% Parameters만 학습해 출력 형식과 행동을 빠르게 바꿨습니다(형식 오류 296 → 0). 하지만 모델이 입력에서 읽어 내지 못하는 정보(격자 속 좌표)까지 만들어 주지는 못했고, 입력을 읽기 쉬운 형태로 바꿨을 때 비로소 과제 성능이 올랐습니다.

코드, 실행 출력, 설명과 분석을 모두 포함한 `.ipynb` 파일 하나를 제출합니다. Adapter, ZIP, CSV 등은 별도로 제출하지 않습니다. 실행 출력이 유지된 상태로 저장합니다.